<a href="https://colab.research.google.com/github/LaxusTiss/Truy_xuat_thong_tin/blob/main/Downloading_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cài đặt thư viện


In [ ]:
!git config --global user.name "LaxusTiss"
!git config --global user.email "taitan2202@gmail.com"

!git init
!git branch -M main
!git remote add origin https://github.com/LaxusTiss/Truy_xuat_thong_tin.git

Reinitialized existing Git repository in /content/.git/
error: remote origin already exists.


In [ ]:
!git remote set-url origin https://github.com/LaxusTiss/Truy_xuat_thong_tin.git

In [ ]:
!pwd
!ls -lah

/content
total 20K
drwxr-xr-x 1 root root 4.0K Nov 24 15:54 .
drwxr-xr-x 1 root root 4.0K Nov 24 15:14 ..
drwxr-xr-x 4 root root 4.0K Nov 18 15:58 .config
drwxr-xr-x 7 root root 4.0K Nov 24 15:55 .git
drwxr-xr-x 1 root root 4.0K Nov 18 15:58 sample_data


In [ ]:
!pip install -q sentence-transformers faiss-cpu huggingface_hub

In [ ]:
import pandas as pd
import numpy as np
from huggingface_hub import hf_hub_download

REPO = "nirajandhakal/goodreads-book-recommend"

def load(name):
    path = hf_hub_download(REPO, name, repo_type="dataset")
    return pd.read_csv(path)

books = load("books.csv")
tags = load("tags.csv")
book_tags = load("book_tags.csv")

print(books.shape)
print(list(books.columns))

books.csv:   0%|          | 0.00/3.29M [00:00<?, ?B/s]

tags.csv:   0%|          | 0.00/722k [00:00<?, ?B/s]

book_tags.csv: reconstructing file:   0%|          |  0.00B / 16.7MB            

book_tags.csv: downloading bytes:           |  0.00B            

(10000, 23)
['book_id', 'goodreads_book_id', 'best_book_id', 'work_id', 'books_count', 'isbn', 'isbn13', 'authors', 'original_publication_year', 'original_title', 'title', 'language_code', 'average_rating', 'ratings_count', 'work_ratings_count', 'work_text_reviews_count', 'ratings_1', 'ratings_2', 'ratings_3', 'ratings_4', 'ratings_5', 'image_url', 'small_image_url']


In [ ]:
display(books[["title", "authors", "average_rating"]].head())
display(tags.head())
display(book_tags.head())

,title,authors,average_rating
0,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,4.34
1,Harry Potter and the Sorcerer's Stone (Harry P...,"J.K. Rowling, Mary GrandPré",4.44
2,"Twilight (Twilight, #1)",Stephenie Meyer,3.57
3,To Kill a Mockingbird,Harper Lee,4.25
4,The Great Gatsby,F. Scott Fitzgerald,3.89


,tag_id,tag_name
0,0,-
1,1,--1-
2,2,--10-
3,3,--12-
4,4,--122-


,goodreads_book_id,tag_id,count
0,1,30574,167697
1,1,11305,37174
2,1,11557,34173
3,1,8717,12986
4,1,33114,12716


In [ ]:
JUNK = {"to-read", "currently-reading", "favorites", "owned",
        "books-i-own", "default", "my-books", "library", "kindle",
        "favourites", "to-buy", "owned-books", "i-own", "wishlist"}

bt = book_tags.merge(tags, on="tag_id")
bt = bt[~bt["tag_name"].isin(JUNK)]
bt = bt.sort_values("count", ascending=False)
bt = bt.groupby("goodreads_book_id").head(8)

tag_text = bt.groupby("goodreads_book_id")["tag_name"].apply(", ".join)
tag_text.head()

,tag_name
goodreads_book_id,
1,"fantasy, young-adult, fiction, harry-potter, y..."
2,"fantasy, children, children-s, all-time-favori..."
3,"fantasy, young-adult, fiction, harry-potter, y..."
5,"fantasy, young-adult, fiction, harry-potter, y..."
6,"fantasy, young-adult, fiction, harry-potter, y..."


In [ ]:
books = books.merge(tag_text.rename("tags"),
                    left_on="goodreads_book_id", right_index=True, how="left")
books["tags"] = books["tags"].fillna("")

books["doc"] = ("Title: " + books["title"].astype(str)
                + ". Author: " + books["authors"].astype(str)
                + ". Tags: " + books["tags"])

print(books["doc"].iloc[0])
print(books["doc"].iloc[1])

Title: The Hunger Games (The Hunger Games, #1). Author: Suzanne Collins. Tags: young-adult, fiction, dystopian, dystopia, fantasy, ya, science-fiction, sci-fi
Title: Harry Potter and the Sorcerer's Stone (Harry Potter, #1). Author: J.K. Rowling, Mary GrandPré. Tags: fantasy, young-adult, fiction, harry-potter, ya, series, magic, childrens
